### Experiment with GPT 4.1-namo

In [3]:
import pandas as pd
import libs.prompts as prompts
from libs.exp_utils import classify_dataset, evaluate_experiment
from libs.openai import call_openai
from tqdm import tqdm
import json
import os

In [4]:
# Define model name
MODEL_NAME = "gpt-4.1-nano"

### Dataset load

In [5]:
# Load ground truth bug locations
with open("../../Datasets/bugLocationDappScan.json", "r") as f:
    bug_locations = json.load(f)

In [6]:
def normalize_path(path):
    if isinstance(path, str):
        return path.replace('\\', '/').replace('./', '').replace('.\\', '')
    return path

bugloc_dict = {
    normalize_path(item['file']): normalize_path(item['location'])
    for item in bug_locations
}


# Load the dataset and convert to appropriate data types
csv_path = "../../Datasets/dw.csv"
df = pd.read_csv(csv_path)
df['has_error'] = df['has_error'].astype(bool)
subset = df.reset_index(drop=True)
#subset.head()

#### Zero-shot Prompting

In [8]:
results_df = classify_dataset(subset, prompts.create_zeroshot_locate_prompt, MODEL_NAME)
precision, recall, f1, conf_matrix = evaluate_experiment(results_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")
results_df.head()

100it [02:42,  1.62s/it]

Precision: 91.89
Recall: 41.98
F1-Score: 57.63
Confusion Matrix:
[[16  3]
 [47 34]]


,predicted_has_error,actual_has_error,bug_type
0,1,0,Fallback function that modifies state variable...
1,0,0,NONE
2,0,0,NONE
3,0,0,NONE
4,0,0,NONE


In [9]:
results_df['file_path'] = subset['file_path']
results_df['file_name'] = subset['file_name'] 

error_cases = results_df[(results_df['predicted_has_error'] == True) & (results_df['actual_has_error'] == True)].copy()

error_cases.head()

,predicted_has_error,actual_has_error,bug_type,file_path,file_name
22,1,1,Integer Overflow / Underflow,DAppSCAN-main/DAppSCAN-source/contracts/openze...,HackerGold.sol
28,1,1,Potential Integer Overflow in getHomeFee and g...,DAppSCAN-main/DAppSCAN-source/contracts/Smartd...,BaseFeeManager.sol
30,1,1,SWC-135-Code With No Effects: The function _pr...,DAppSCAN-main/DAppSCAN-source/contracts/Iosiro...,Exchanger.sol
32,1,1,Function _mint is defined as internal and over...,DAppSCAN-main/DAppSCAN-source/contracts/PeckSh...,LuckyToken.sol
34,1,1,SWC-105-Unprotected Ether Withdrawal,DAppSCAN-main/DAppSCAN-source/contracts/consen...,Moloch.sol


In [10]:
def get_gt_category(row):
    file_path = row['file_path'].replace('DAppSCAN-main/DAppSCAN-source/', '')
    if file_path in bugloc_dict:
        location_json_path = os.path.join('../../Datasets/DAppSCAN-main/DAppSCAN-source/', bugloc_dict[file_path])
        if os.path.exists(location_json_path):
            with open(location_json_path, 'r') as f:
                location_data = json.load(f)
                swcs = location_data.get('SWCs', [])
                categories = [swc['category'] for swc in swcs if 'category' in swc]
                return categories if categories else ['Unknown']
        else:
            return ['JSON não encontrado']
    else:
        return ['Arquivo não encontrado']

In [13]:
# Roda a checagem
llm_categories = []
gt_categories = []

for idx, row in error_cases.iterrows():
    print(f"\n--- Rodando para idx {idx} ---")
    gt_cat = get_gt_category(row)

    llm_categories.append(row['bug_type'])
    gt_categories.append(gt_cat)
    print(f"File: {row['file_path']}\nLLM: {row['bug_type']}\nGT: {gt_cat}\n")




--- Rodando para idx 22 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/openzeppelin-EtherCamp_Hacker_Gold/virtual-accelerator-2529ffe5efd5294b44f1bc89dc9a4721a7b16409/HackerGold.sol
LLM: Integer Overflow / Underflow
GT: ['SWC-101-Integer Overflow and Underflow', 'SWC-135-Code With No Effects', 'SWC-135-Code With No Effects']


--- Rodando para idx 28 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/Smartdec-TokenBridge (by POA Network) Smart Contracts Security Analysis/tokenbridge-contracts-bbb97a63c900f03a902d0e82358abac3b294e4d9/contracts/upgradeable_contracts/BaseFeeManager.sol
LLM: Potential Integer Overflow in getHomeFee and getForeignFee functions
GT: ['SWC-110-Assert Violation']


--- Rodando para idx 30 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/Iosiro-Synthetix Alnitak Release Smart Contract Audit/synthetix-00df930622a8ea462d15d4eccf6bc6b3d704bd21/contracts/Exchanger.sol
LLM: SWC-135-Code With No Effects: The function _processTradingRewards contains an 'if' stateme

In [14]:
comparison_results = []

for bug_pred, gt_cat in tqdm(zip(error_cases['bug_type'], gt_categories), total=len(gt_categories)):
    prompt = prompts.create_zeroshot_error_comparison_prompt(bug_pred, gt_cat)
    response = call_openai(MODEL_NAME, prompt)
    try:
        result = json.loads(response)
        same_error = int(result.get("same_error", False))
    except Exception as e:
        print("Erro ao interpretar resposta da LLM:", e)
        same_error = 0 

    comparison_results.append({'actual_has_error': 1, 'predicted_has_error': same_error})

comparison_df = pd.DataFrame(comparison_results)

precision, recall, f1, conf_matrix = evaluate_experiment(comparison_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")


  0%|          | 0/34 [00:00<?, ?it/s]

100%|██████████| 34/34 [00:30<00:00,  1.11it/s]

Precision: 100.00
Recall: 52.94
F1-Score: 69.23
Confusion Matrix:
[[ 0  0]
 [16 18]]


#### Zero-shot CoT Prompting

In [15]:
results_df = classify_dataset(subset, prompts.create_zeroshot_cot_locate_prompt, MODEL_NAME)
precision, recall, f1, conf_matrix = evaluate_experiment(results_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")
results_df.head()

100it [05:33,  3.33s/it]

Precision: 86.67
Recall: 32.10
F1-Score: 46.85
Confusion Matrix:
[[15  4]
 [55 26]]


,predicted_has_error,actual_has_error,bug_type
0,0,0,NONE
1,0,0,NONE
2,0,0,NONE
3,0,0,NONE
4,0,0,NONE


In [16]:
results_df['file_path'] = subset['file_path']
results_df['file_name'] = subset['file_name'] 

error_cases = results_df[(results_df['predicted_has_error'] == True) & (results_df['actual_has_error'] == True)].copy()

error_cases.head()

,predicted_has_error,actual_has_error,bug_type,file_path,file_name
22,1,1,Integer Overflow and Underflow,DAppSCAN-main/DAppSCAN-source/contracts/openze...,HackerGold.sol
29,1,1,Reentrancy vulnerability,DAppSCAN-main/DAppSCAN-source/contracts/Smartd...,HomeBridgeErcToErc.sol
34,1,1,Unprotected Ether Withdrawal (SWC-105),DAppSCAN-main/DAppSCAN-source/contracts/consen...,Moloch.sol
35,1,1,Potential Reentrancy / Approval Race Condition,DAppSCAN-main/DAppSCAN-source/contracts/Inspex...,SpookySwapStrategyWithdrawMinimizeTrading.sol
38,1,1,Arithmetic Overflow/Underflow in Calculation,DAppSCAN-main/DAppSCAN-source/contracts/Inspex...,SpookySwapStrategyAddBaseTokenOnly.sol


In [17]:
def get_gt_category(row):
    file_path = row['file_path'].replace('DAppSCAN-main/DAppSCAN-source/', '')
    if file_path in bugloc_dict:
        location_json_path = os.path.join('../../Datasets/DAppSCAN-main/DAppSCAN-source/', bugloc_dict[file_path])
        if os.path.exists(location_json_path):
            with open(location_json_path, 'r') as f:
                location_data = json.load(f)
                swcs = location_data.get('SWCs', [])
                categories = [swc['category'] for swc in swcs if 'category' in swc]
                return categories if categories else ['Unknown']
        else:
            return ['JSON não encontrado']
    else:
        return ['Arquivo não encontrado']

In [19]:
# Roda a checagem
llm_categories = []
gt_categories = []

for idx, row in error_cases.iterrows():
    print(f"\n--- Rodando para idx {idx} ---")
    gt_cat = get_gt_category(row)

    llm_categories.append(row['bug_type'])
    gt_categories.append(gt_cat)
    print(f"File: {row['file_path']}\nLLM: {row['bug_type']}\nGT: {gt_cat}\n")


--- Rodando para idx 22 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/openzeppelin-EtherCamp_Hacker_Gold/virtual-accelerator-2529ffe5efd5294b44f1bc89dc9a4721a7b16409/HackerGold.sol
LLM: Integer Overflow and Underflow
GT: ['SWC-101-Integer Overflow and Underflow', 'SWC-135-Code With No Effects', 'SWC-135-Code With No Effects']


--- Rodando para idx 29 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/Smartdec-TokenBridge (by POA Network) Smart Contracts Security Analysis/tokenbridge-contracts-bbb97a63c900f03a902d0e82358abac3b294e4d9/contracts/upgradeable_contracts/erc20_to_erc20/HomeBridgeErcToErc.sol
LLM: Reentrancy vulnerability
GT: ['SWC-135-Code With No Effects']


--- Rodando para idx 34 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/consensys-The_LAO/moloch-4bc443f4dad60279b47978fc6987bb978d3dfc58/contracts/Moloch.sol
LLM: Unprotected Ether Withdrawal (SWC-105)
GT: ['SWC-114-Transaction Order Dependence', 'SWC-105-Unprotected Ether Withdrawal', 'SWC-101-Integer Overflow a

In [20]:
comparison_results = []

for bug_pred, gt_cat in tqdm(zip(error_cases['bug_type'], gt_categories), total=len(gt_categories)):
    prompt = prompts.create_zeroshot_error_comparison_prompt(bug_pred, gt_cat)
    response = call_openai(MODEL_NAME, prompt)
    try:
        result = json.loads(response)
        same_error = int(result.get("same_error", False))
    except Exception as e:
        print("Erro ao interpretar resposta da LLM:", e)
        same_error = 0 

    comparison_results.append({'actual_has_error': 1, 'predicted_has_error': same_error})

comparison_df = pd.DataFrame(comparison_results)

precision, recall, f1, conf_matrix = evaluate_experiment(comparison_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")

100%|██████████| 26/26 [00:26<00:00,  1.01s/it]

Precision: 100.00
Recall: 61.54
F1-Score: 76.19
Confusion Matrix:
[[ 0  0]
 [10 16]]


#### Zero-shot ToT Prompting

In [10]:
results_df = classify_dataset(subset, prompts.create_zeroshot_tot_locate_prompt, MODEL_NAME)
precision, recall, f1, conf_matrix = evaluate_experiment(results_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")
results_df.head()

400it [1:16:32, 11.48s/it]

Precision: 58.93
Recall: 99.00
F1-Score: 73.88
Confusion Matrix:
[[ 62 138]
 [  2 198]]


,predicted_has_error,actual_has_error,bug_type,location
0,1,0,Unprotected fallback function allowing arbitra...,48-51
1,1,0,Arbitrary Storage Write Vulnerability,64-68
2,1,0,Multiple inheritance initialization conflict,94-95
3,1,0,Incorrect bitmask usage and missing access con...,26-34
4,1,0,Lack of access control allowing arbitrary stor...,"18-21, 28-31, 38-41, 48-51, 58-61, 68-71"


In [11]:
results_df['file_path'] = subset['file_path']
results_df['file_name'] = subset['file_name'] 

error_cases = results_df[(results_df['predicted_has_error'] == True) & (results_df['actual_has_error'] == True)].copy()

error_cases.head()

,predicted_has_error,actual_has_error,bug_type,location,file_path,file_name
200,1,1,NONE,NONE,DAppSCAN-main/DAppSCAN-source/contracts/QuillA...,ERC20Permit.sol
201,1,1,Floating Pragma,3,DAppSCAN-main/DAppSCAN-source/contracts/Chains...,AccessControl.sol
202,1,1,Floating Pragma and Outdated Compiler Version,4,DAppSCAN-main/DAppSCAN-source/contracts/Chains...,AccessControl.sol
203,1,1,Integer Overflow/Underflow,"30, 90",DAppSCAN-main/DAppSCAN-source/contracts/openze...,HackerGold.sol
204,1,1,Integer Underflow leading to logical error,121-124,DAppSCAN-main/DAppSCAN-source/contracts/PeckSh...,UserConfiguration.sol


In [12]:
def get_gt_category(row):
    file_path = row['file_path'].replace('DAppSCAN-main/DAppSCAN-source/', '')
    if file_path in bugloc_dict:
        location_json_path = os.path.join('../../Datasets/DAppSCAN-main/DAppSCAN-source/', bugloc_dict[file_path])
        if os.path.exists(location_json_path):
            with open(location_json_path, 'r') as f:
                location_data = json.load(f)
                swcs = location_data.get('SWCs', [])
                categories = [swc['category'] for swc in swcs if 'category' in swc]
                return categories if categories else ['Unknown']
        else:
            return ['JSON não encontrado']
    else:
        return ['Arquivo não encontrado']

In [13]:
# Roda a checagem
llm_categories = []
gt_categories = []

for idx, row in error_cases.iterrows():
    print(f"\n--- Rodando para idx {idx} ---")
    gt_cat = get_gt_category(row)

    llm_categories.append(row['bug_type'])
    llm_categories.append(row['location'])  # Assuming 'bug_type' is the LLM's prediction
    gt_categories.append(gt_cat)
    print(f"File: {row['file_path']}\nLLM: {row['bug_type']}\nGT: {gt_cat}\n")

    print("LOCATION: LINE - ", row['location'])
#print("LLM Categories:", llm_categories, "GT Categories:", gt_categories)



--- Rodando para idx 200 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/QuillAudits-1inch-token/1inch-token-99fd056f91005ca521a02a005f7bcd8f77e06afc/contracts/ERC20Permit.sol
LLM: NONE
GT: ['SWC-114-Transaction Order Dependence']

LOCATION: LINE -  NONE

--- Rodando para idx 201 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/Chainsulting-SWAPP Protocol-project1/openzeppelin-contracts-3.3.0/contracts/access/AccessControl.sol
LLM: Floating Pragma
GT: ['SWC-103-Floating Pragma']

LOCATION: LINE -  3

--- Rodando para idx 202 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/Chainsulting-GSPI Club-project2/openzeppelin-contracts-3.2.0/contracts/access/AccessControl.sol
LLM: Floating Pragma and Outdated Compiler Version
GT: ['SWC-103-Floating Pragma', 'SWC-102-Outdated Compiler Version']

LOCATION: LINE -  4

--- Rodando para idx 203 ---
File: DAppSCAN-main/DAppSCAN-source/contracts/openzeppelin-EtherCamp_Hacker_Gold/virtual-accelerator-2529ffe5efd5294b44f1bc89dc9a4721a7b16409/Hacker

In [14]:
comparison_results = []

for bug_pred, gt_cat in tqdm(zip(error_cases['bug_type'], gt_categories), total=len(gt_categories)):
    prompt = prompts.create_zeroshot_error_comparison_prompt(bug_pred, gt_cat)
    response = call_openai(MODEL_NAME, prompt)
    try:
        result = json.loads(response)
        same_error = int(result.get("same_error", False))
    except Exception as e:
        print("Erro ao interpretar resposta da LLM:", e)
        same_error = 0 

    comparison_results.append({'actual_has_error': 1, 'predicted_has_error': same_error})

comparison_df = pd.DataFrame(comparison_results)

precision, recall, f1, conf_matrix = evaluate_experiment(comparison_df)
print(f"Precision: {precision:.2f}")
print(f"Recall: {recall:.2f}")
print(f"F1-Score: {f1:.2f}")
print(f"Confusion Matrix:\n{conf_matrix}")

100%|██████████| 198/198 [02:57<00:00,  1.11it/s]

Precision: 100.00
Recall: 54.55
F1-Score: 70.59
Confusion Matrix:
[[  0   0]
 [ 90 108]]
